<a href="https://colab.research.google.com/github/stellatws2010/lottery-sort-demo/blob/main/%E7%B5%A6%E6%9D%B1%E9%9C%96%E8%AA%BF%E6%95%B4%E7%94%A8lottery_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎱 樂透中獎機率模擬
使用 Monte Carlo 模擬法計算各種對中球數的平均人數

In [ ]:
import random
import numpy as np
from collections import defaultdict

# ==================== 變數設定 ====================
NUM_PLAYERS       = 19    # 玩家人數
BALL_RANGE        = 30    # 樂透球號範圍 (1 ~ BALL_RANGE)
TICKETS_PER_PLAYER = 1   # 每人簽幾柱
BALLS_PER_TICKET  = 6    # 每注幾球
DRAWN_BALLS       = 8    # 主持人抽幾個球
SIMULATIONS       = 100  # 模擬次數
# =================================================

def simulate_lottery(num_players, ball_range, tickets_per_player,
                     balls_per_ticket, drawn_balls, simulations):
    """
    模擬樂透抽獎，回傳每種對中球數的累計人次。
    """
    # 記錄每局各對中球數的人數
    match_counts = defaultdict(list)  # {match_num: [人數_第1局, 人數_第2局, ...]}

    for sim in range(simulations):
        # 主持人從 1~ball_range 中抽出 drawn_balls 個球
        winning_balls = set(random.sample(range(1, ball_range + 1), drawn_balls))

        # 統計本局各對中球數的人數
        sim_match = defaultdict(int)

        for player in range(num_players):
            for ticket in range(tickets_per_player):
                # 每注從 1~ball_range 選 balls_per_ticket 個號碼
                player_ticket = set(random.sample(range(1, ball_range + 1), balls_per_ticket))
                # 計算對中幾球
                matched = len(player_ticket & winning_balls)
                sim_match[matched] += 1

        # 把本局結果存入
        for m in range(balls_per_ticket + 1):  # 0 ~ balls_per_ticket
            match_counts[m].append(sim_match[m])

    return match_counts


# 執行模擬
results = simulate_lottery(
    NUM_PLAYERS, BALL_RANGE, TICKETS_PER_PLAYER,
    BALLS_PER_TICKET, DRAWN_BALLS, SIMULATIONS
)

# ==================== 輸出結果 ====================
print('=' * 45)
print(f'  🎱 樂透中獎機率模擬結果 (共 {SIMULATIONS} 局)')
print('=' * 45)
print(f'  玩家人數        : {NUM_PLAYERS}')
print(f'  球號範圍        : 1 ~ {BALL_RANGE}')
print(f'  每人簽幾柱      : {TICKETS_PER_PLAYER}')
print(f'  每注幾球        : {BALLS_PER_TICKET}')
print(f'  主持人抽幾個球  : {DRAWN_BALLS}')
print('=' * 45)
print(f'{"對中球數":^10} {"平均人數":^12} {"最少":^6} {"最多":^6}')
print('-' * 45)

for match in range(BALLS_PER_TICKET, -1, -1):
    data = results[match]
    avg  = np.mean(data)
    mn   = np.min(data)
    mx   = np.max(data)
    print(f'  對中 {match} 球    {avg:^12.4f} {mn:^6} {mx:^6}')

print('=' * 45)
print('✅ 模擬完成！')

  🎱 樂透中獎機率模擬結果 (共 100 局)
  玩家人數        : 19
  球號範圍        : 1 ~ 30
  每人簽幾柱      : 1
  每注幾球        : 6
  主持人抽幾個球  : 8
   對中球數        平均人數       最少     最多  
---------------------------------------------
  對中 6 球       0.0000      0      0   
  對中 5 球       0.1000      0      2   
  對中 4 球       0.5300      0      3   
  對中 3 球       2.6500      0      7   
  對中 2 球       6.4900      2      12  
  對中 1 球       6.8400      2      12  
  對中 0 球       2.3900      0      6   
✅ 模擬完成！


## 💰 獎金期望值計算
根據模擬出的平均對中人數，計算每局各等級的總獎金期望值，以及每位玩家的個人期望獎金。

In [ ]:
# ==================== 獎金設定 ====================
PRIZE = {
    6: 60000,   # 對中 6 球
    5: 30000,   # 對中 5 球
    4: 15000,   # 對中 4 球
    3:  8000,   # 對中 3 球
    2:  5000,   # 對中 2 球
    1:  4000,   # 對中 1 球
    0:  3000,   # 對中 0 球
}
# =================================================

# 從模擬結果取得各對中球數的平均人數
avg_people = {m: np.mean(results[m]) for m in range(BALLS_PER_TICKET + 1)}

# 計算總票數（全部玩家 × 每人柱數）
total_tickets = NUM_PLAYERS * TICKETS_PER_PLAYER

# 每張票的中獎機率 = 平均對中人數 / 總票數
prob = {m: avg_people[m] / total_tickets for m in avg_people}

# 個人期望獎金 = Σ (對中機率 × 獎金)
personal_ev = sum(prob[m] * PRIZE[m] for m in PRIZE)

# 每局總獎金期望值 = Σ (平均對中人數 × 獎金)
total_ev = sum(avg_people[m] * PRIZE[m] for m in PRIZE)

# ==================== 輸出結果 ====================
print('=' * 58)
print(f'  💰 獎金期望值分析 (共 {SIMULATIONS} 局模擬)')
print('=' * 58)
print(f'  {'對中球數':<8} {'獎金':>8}  {'平均中獎人數':>12}  {'該等級總獎金期望':>14}')
print('-' * 58)

for m in range(BALLS_PER_TICKET, -1, -1):
    prize     = PRIZE[m]
    avg_cnt   = avg_people[m]
    level_ev  = avg_cnt * prize
    prize_str = f'${prize:,}' if prize > 0 else '—'
    print(f'  對中 {m} 球    {prize_str:>8}  {avg_cnt:>12.4f}  {level_ev:>14,.1f}')

print('=' * 58)
print(f'  📦 每局總獎金期望值（全體）  : ${total_ev:>12,.2f}')
print(f'  🙋 每位玩家個人期望獎金      : ${personal_ev:>12,.2f}')
print('=' * 58)
print('✅ 期望值計算完成！')

  💰 獎金期望值分析 (共 100 局模擬)
  對中球數           獎金        平均中獎人數        該等級總獎金期望
----------------------------------------------------------
  對中 6 球     $60,000        0.0000             0.0
  對中 5 球     $30,000        0.1000         3,000.0
  對中 4 球     $15,000        0.5300         7,950.0
  對中 3 球      $8,000        2.6500        21,200.0
  對中 2 球      $5,000        6.4900        32,450.0
  對中 1 球      $4,000        6.8400        27,360.0
  對中 0 球      $3,000        2.3900         7,170.0
  📦 每局總獎金期望值（全體）  : $   99,130.00
  🙋 每位玩家個人期望獎金      : $    5,217.37
✅ 期望值計算完成！
